In [129]:
import pandas as pd
import numpy as np

import pickle

from azureml.opendatasets import NoaaIsdWeather
from datetime import datetime, timedelta

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

from azureml.core import Workspace
from azureml.core.compute import AksCompute, ComputeTarget
from azureml.core.webservice import Webservice, AksWebservice
from azureml.core.model import Model

from azureml.core import Dataset

In [114]:
data = pd.read_csv('data/training.csv')
data = data.fillna(0)

In [115]:
data.head(5)

,Unnamed: 0,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,cloudCoverage,precipTime,precipDepth,snowDepth,stationName,countryOrRegion,p_k
0,6537683,720735,73805,2019-01-01 00:00:00,30.349,-85.788,21.0,140.0,5.1,21.1,0.0,0,0.0,0.0,0.0,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805
1,6537684,720735,73805,2019-01-01 00:39:00,30.349,-85.788,21.0,150.0,5.7,21.1,0.0,0,0.0,0.0,0.0,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805
2,6537685,720735,73805,2019-01-01 00:53:00,30.349,-85.788,21.0,150.0,4.6,21.1,1019.5,0,1.0,0.0,0.0,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805
3,6537686,720735,73805,2019-01-01 01:01:00,30.349,-85.788,21.0,150.0,4.6,21.1,0.0,0,0.0,0.0,0.0,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805
4,6537687,720735,73805,2019-01-01 01:53:00,30.349,-85.788,21.0,140.0,4.1,21.1,1019.6,0,1.0,0.0,0.0,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805


In [116]:
lr = LinearRegression()

In [117]:
X = data[['usaf', 'wban', 'latitude', 'longitude', 'elevation', 'temperature']]
y = np.array(data['windSpeed'])

In [118]:
lr = lr.fit(X, y)

In [119]:
y_pred = lr.predict(X)

In [120]:
mean_squared_error(y, y_pred)

4.535010902831102

In [121]:
with open('model.pkl', 'wb') as f:
    pickle.dump(lr, f)

In [122]:
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

In [123]:
isd = NoaaIsdWeather(datetime.now() - timedelta(days=5), datetime.now()).to_pandas_dataframe()

ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2019/month=9/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2019/month=9/part-00089-tid-8804999204571197097-df7ddc8a-f146-41e2-89ef-094cff0e4923-202392.c000.snappy.parquet under container isdweatherdatacontainer
Done.
ActivityCompleted: Activity=to_pandas_dataframe_in_worker, HowEnded=Success, Duration=18423.12 [ms]
ActivityCompleted: Activity=to_pandas_dataframe, HowEnded=Success, Duration=18443.25 [ms]


In [124]:
test_data = isd[isd['stationName'].str.contains('FLORIDA', regex=True, na=False)]
test_data = test_data.fillna(0)

In [125]:
X_test = test_data[['usaf', 'wban', 'latitude', 'longitude', 'elevation', 'temperature']]
y_test = np.array(test_data['windSpeed'])

In [126]:
y_pred = model.predict(X_test)

In [128]:
mean_squared_error(y_test, y_pred)

8.256827044818303

In [130]:
ws = Workspace.from_config()


In [131]:
dstore = ws.get_default_datastore()
dstore.upload('data', 'my-data', overwrite=True)

Uploading an estimated of 1 files
Uploading data/training.csv
Uploaded data/training.csv, 1 files out of an estimated total of 1
Uploaded 1 files


$AZUREML_DATAREFERENCE_65503b610f234ccdb9f35b0e3bb9b621

In [132]:
dset = Dataset.Tabular.from_delimited_files(dstore.path('my-data/training.csv'))

In [133]:
dset.register(ws, 'my-dataset')

TabularDataset
{
  "source": [
    "('workspaceblobstore', 'my-data/training.csv')"
  ],
  "definition": [
    "GetDatastoreFiles",
    "ParseDelimited",
    "DropColumns",
    "SetColumnTypes"
  ],
  "registration": {
    "name": "my-dataset",
    "version": 1,
    "workspace": "Workspace.create(name='Azure-ML', subscription_id='60582a10-b9fd-49f1-a546-c4194134bba8', resource_group='copetersRG')"
  }
}

In [134]:
model = Model.register(model_path = 'model.pkl', # this points to a local file
                       model_name = 'my-model', # this is the name the model is registered as
                       tags = {'area': 'weather', 'type': 'regression'},
                       description = 'Simple model for predicting temperature based on other weather details. Trained on data from Florida in the winter',
                       workspace = ws,
                       datasets  = [('training_data', dset)])

print(model.name, model.description, model.version, sep='\n')

Registering model my-model
my-model Simple model for predicting temperature based on other weather details. Trained on data from Florida in the winter 1


In [146]:
from azureml.core import Environment
from azureml.core.conda_dependencies import CondaDependencies 

conda_deps = CondaDependencies.create(conda_packages=['numpy','scikit-learn'], pip_packages=['azureml-defaults'])
myenv = Environment(name='myenv')
myenv.python.conda_dependencies = conda_deps

In [162]:
%%writefile score.py
import pickle
import json
import numpy
from sklearn.externals import joblib
from sklearn.linear_model import Ridge
from azureml.core.model import Model

def init():
    global model
    # note here "sklearn_regression_model.pkl" is the name of the model registered under
    # this is a different behavior than before when the code is run locally, even though the code is the same.
    model_path = Model.get_model_path('my-model')
    # deserialize the model file back into a sklearn model
    model = joblib.load(model_path)

# note you can pass in multiple rows for scoring
def run(raw_data):
    try:
        data = json.loads(raw_data)['data']
        data = numpy.array(data)
        result = model.predict(data)
        # you can return any data type as long as it is JSON-serializable
        return result.tolist()
    except Exception as e:
        error = str(e)
        return error

Overwriting score.py


In [163]:
from azureml.core.model import InferenceConfig

inf_config = InferenceConfig(entry_script='score.py', environment=myenv)

In [164]:
# Use the default configuration (can also provide parameters to customize)
prov_config = AksCompute.provisioning_configuration()

aks_name = 'temperature-aks' 
# Use existing cluster or create new cluster 
try:
    aks_target = ComputeTarget(workspace=ws, name=aks_name)
except:
    aks_target = ComputeTarget.create(workspace = ws, name = aks_name, provisioning_configuration=prov_config)

In [165]:
%%time
#aks_target.wait_for_completion(show_output = True)
print(aks_target.provisioning_state)
print(aks_target.provisioning_errors)

Succeeded
None
CPU times: user 0 ns, sys: 0 ns, total: 0 ns
Wall time: 453 µs


In [166]:
# Set the web service configuration (using default here)
aks_config = AksWebservice.deploy_configuration()

In [167]:
%%time
aks_service_name ='my-service'

aks_service = Model.deploy(workspace=ws,
                           name=aks_service_name,
                           models=[model],
                           inference_config=inf_config,
                           deployment_config=aks_config,
                           deployment_target=aks_target)

aks_service.wait_for_deployment(show_output = True)
print(aks_service.state)

Running...
SucceededAKS service creation operation finished, operation "Succeeded"
Healthy
CPU times: user 266 ms, sys: 1.28 s, total: 1.55 s
Wall time: 28.7 s


In [175]:
X_test.values.tolist()

[['854880', '99999', -29.916, -71.2, 147.0, 12.0],
 ['854880', '99999', -29.917, -71.2, 146.0, 11.7],
 ['854880', '99999', -29.916, -71.2, 147.0, 12.0],
 ['854880', '99999', -29.916, -71.2, 147.0, 12.0],
 ['854880', '99999', -29.916, -71.2, 147.0, 13.0],
 ['854880', '99999', -29.917, -71.2, 146.0, 14.5],
 ['854880', '99999', -29.916, -71.2, 147.0, 15.0],
 ['854880', '99999', -29.916, -71.2, 147.0, 15.0],
 ['854880', '99999', -29.916, -71.2, 147.0, 15.0],
 ['854880', '99999', -29.917, -71.2, 146.0, 13.7],
 ['854880', '99999', -29.916, -71.2, 147.0, 14.0],
 ['854880', '99999', -29.916, -71.2, 147.0, 13.0],
 ['854880', '99999', -29.916, -71.2, 147.0, 12.0],
 ['854880', '99999', -29.917, -71.2, 146.0, 11.0],
 ['854880', '99999', -29.916, -71.2, 147.0, 11.0],
 ['854880', '99999', -29.916, -71.2, 147.0, 11.0],
 ['854880', '99999', -29.916, -71.2, 147.0, 11.0],
 ['854880', '99999', -29.917, -71.2, 146.0, 11.2],
 ['854880', '99999', -29.916, -71.2, 147.0, 11.0],
 ['854880', '99999', -29.916, -

In [179]:
%%time
import json

test_sample = json.dumps({'data': X_test.to_json()})
test_sample = bytes(test_sample,encoding = 'utf8')

prediction = aks_service.run(input_data = test_sample)
print(prediction)

Expected 2D array, got scalar array instead:
array={"usaf":{"2914551":"854880","2914552":"854880","2914553":"854880","2914554":"854880","2914555":"854880","2914556":"854880","2914557":"854880","2914558":"854880","2914559":"854880","2914560":"854880","2914561":"854880","2914562":"854880","2914563":"854880","2914564":"854880","2914565":"854880","2914566":"854880","2914567":"854880","2914568":"854880","2914569":"854880","2914570":"854880","2914571":"854880","2914572":"854880","2914573":"854880","2914574":"854880","2914575":"854880","2914576":"854880","2914577":"854880","2914578":"854880","2914579":"854880","2914580":"854880","2914581":"854880","2914582":"854880","2914583":"854880","2914584":"854880","2914585":"854880","2914586":"854880","2914587":"854880","2914588":"854880","2914589":"854880","2914590":"854880","2914591":"854880","2914592":"854880","2914593":"854880","2914594":"854880","2914595":"854880","2914596":"854880","2914597":"854880","2914598":"854880","2914599":"854880","2914600"

In [161]:
#%%time
aks_service.delete()
#model.delete()